## **Arabic Named Entity Recognition**
Named entity recognition (NER) is the task of finding spans of text that constitute proper names and labeling the type of such entities.
As we discussed in the lecture, this task can be formulated as a sequence labeling problem, e.g., using the BIO tagging scheme.
Your task is to build a NER tagger for Arabic using Conditional Random Fields (CRFs) and explore different sets of features to achieve improvement over the baseline tagger.
Arabic is a morphologically rich language, which poses certain challenges to NLP.
We use ANERcorp that has 150K words annotated for four entities: Location (LOC), Organization (ORG), and Person (PER), and Miscellaneous (MISC).
Since ANERcorp only has train and test splits, we take 10% of the train as a development set.


## **The Task**
Your task is to improve the feature function (`word2features()`).
Currently, the baseline model only takes into account previous and next words.
You may use external tools to obtain useful features such as POS and morphological tags.
You will also need to submit a report describing your method, result, and analysis (1-2 pages).

## **Data Preparation**

In [ ]:
%pip install scikit-learn seqeval
%pip install python-crfsuite
!test -f ANERcorp-CamelLabSplits.zip || wget https://camel.abudhabi.nyu.edu/anercorp/ANERcorp-CamelLabSplits.zip
!unzip -o ANERcorp-CamelLabSplits.zip -x "__MACOSX/*"

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

# read ANER named entity dataset
def read_data(file_path):
    with open(file_path, mode='r') as f:
        sent, sents = [], []
        for line in f.readlines():
            if len(line.strip()) > 0:
                # split the line by space
                token = line.strip().split(' ')
                # unpack the token
                word, ner = token
                # append the tuple (word, ner tag) to a list for one sentence
                sent.append((word, ner.replace('PERS', 'PER')))
            # if the line is empty and the list is not empty
            elif len(sent):
                sents.append(sent)
                sent = []
        if sent:
            sents.append(sent)
    return sents

# read training and test data
_train_sents = read_data('./ANERcorp-CamelLabSplits/ANERCorp_CamelLab_train.txt')
test_sents = read_data('./ANERcorp-CamelLabSplits/ANERCorp_CamelLab_test.txt')

# since this dataset does not have dev set, we take 10% of the train as dev set
train_sents, dev_sents = train_test_split(_train_sents, test_size=0.1, random_state=12345)

## **Feature Engineering**
Here, your task is to design a function that tranforms each word into a set of features.
You may want to use existing tools to get POS, morphological features, etc.
The current model gives `0.70` in $F_1$ on the dev set.

In [21]:
from camel_tools.disambig.mle import MLEDisambiguator
from tqdm import tqdm
from arbtok.tokenizer import Sentence

KEYS = []
ALL_FEATURES = set()

def drop_feat(data: dict):
    return {k: v for k,v in data.items() if k in KEYS}

mle_msa = MLEDisambiguator.pretrained('calima-msa-r13')

cache = {}
def word2features(sent, i):
    global ALL_FEATURES
    word = sent[i][0]
    full_sent = ' '.join([j[0] for j in sent])
    if full_sent not in cache:
        cache[full_sent] = {
            'morphological_mle':  [k for k in mle_msa.disambiguate([j[0] for j in sent])],
            # 'morphological_egy':  [k for k in mle_egy.disambiguate([j[0] for j in sent])]
        }

    features = {
        'word': word,
        'ipa': Sentence(word).ipa,
        # 'diacritize': TashkeelDiacritizer().diacritize(word),
        # 'ar_transcript': G2P("ar").transcribe(word),
    }

    if len(cache[full_sent]['morphological_mle'][i].analyses) > 0:
        feats = cache[full_sent]['morphological_mle'][i].analyses[0].analysis
        features = {
            **features,
            **{k:v for k,v in feats.items()},
            }

    if i > 0:
        prev_word = sent[i-1][0]

        features.update({
            '-1:word': prev_word
        })
    else:
        features['BOS'] = True

    if i < len(sent)-1:
        next_word = sent[i+1][0]

        features.update({
            '+1:word': next_word,
        })
    else:
        features['EOS'] = True

    ALL_FEATURES = ALL_FEATURES | set(features.keys())

    return drop_feat({k: str(v) if v is None else v for k, v  in features.items()})

def sent2features(sent):
    return [word2features(sent, i) for i in range(len(sent))]

def sent2labels(sent):
    return [label for token, label in sent]

def sent2tokens(sent):
    return [token for token, label in sent]

y_train = [sent2labels(s) for s in train_sents]
LABELS = sorted(list(set(sum(y_train, []))))

def performance(y, y_hat):
    y_ = sum(y, [])
    y_hat_ = sum(y_hat, [])
    
    # Construct Confusion Matrix
    confusion_matrix = np.zeros((len(LABELS), len(LABELS)))
    for i, j in zip(y_, y_hat_):
        confusion_matrix[LABELS.index(i),LABELS.index(j)] += 1

    micro_avg = confusion_matrix.diagonal().sum()/confusion_matrix.sum() # P, R, F1
    macro_P = np.array([confusion_matrix[i,i]/confusion_matrix[:,i].sum() for i in range(len(LABELS))])
    macro_R = np.array([confusion_matrix[i,i]/confusion_matrix[i,:].sum() for i in range(len(LABELS))])
    macro_F1 = 2*macro_P*macro_R/(macro_P + macro_R)
    
    return {
        'micro_avg': micro_avg,
        'macro_P': macro_P.mean(),
        'macro_R': macro_R.mean(),
        'macro_F1': macro_F1.mean()
    }

X_train = [sent2features(s) for s in tqdm(train_sents)]
y_train = [sent2labels(s) for s in train_sents]
LABELS = sorted(list(set(sum(y_train, []))))

X_dev = [sent2features(s) for s in tqdm(dev_sents)]
y_dev = [sent2labels(s) for s in dev_sents]

100%|██████████| 398/398 [00:05<00:00, 75.62it/s]


### Hyperparam searching

Random selection with weight, remove features mostly reduce performance on dev set, so maybe the idea is to create more different features

In [ ]:
# # This does not help, still around 80 on dev 
# import pycrfsuite
# import random

# CONSISTENCY_TRIALS = 10
# TRIALS = 40
# TRAIN_SAMPLES = 500
# BEST_PERFORMANCE_SO_FAR = 0
# BEST_SELECTION_SO_FAR = []
# SELECTED_FEATURES = 30

# hyper_train_x = X_train[:TRAIN_SAMPLES]
# hyper_train_y = y_train[:TRAIN_SAMPLES]
# TRIAL_APPEARANCES = []
# for _ in range(CONSISTENCY_TRIALS):
#     FEATURE_SCORES = {k: 0. for k in set.union(*[set(sum(X_train[:100], [])[i].keys()) for i in range(100)])}
#     fairness_ensurance = sum([list(FEATURE_SCORES.keys()) for _ in range(TRIALS * (len(list(FEATURE_SCORES.keys()))- SELECTED_FEATURES) // len(list(FEATURE_SCORES.keys())))], [])
#     random.shuffle(fairness_ensurance)
#     for i in range(TRIALS):
#         selected_feats = fairness_ensurance[:SELECTED_FEATURES]
#         fairness_ensurance = fairness_ensurance[SELECTED_FEATURES:]
#         trial_train_x = [[{k: v for k, v in t.items() if k in selected_feats} for t in xs] for xs in hyper_train_x]
#         trial_dev_x = [[{k: v for k, v in t.items() if k in selected_feats} for t in xs] for xs in X_dev]

#         # training: copy from below
#         hyper_trainer = pycrfsuite.Trainer(verbose=False)
#         for xseq, yseq in zip(trial_train_x, hyper_train_y):
#             hyper_trainer.append(xseq, yseq)
#         hyper_trainer.set_params({
#             'c1': 1.0,   # coefficient for L1 penalty
#             'c2': 1e-3,  # coefficient for L2 penalty
#             'max_iterations': 50,  # stop earlier
#             'feature.possible_transitions': True
#         })
#         hyper_trainer.train('arabic-ner.crfsuite')

#         # dev eval
#         tagger = pycrfsuite.Tagger()
#         tagger.open('arabic-ner.crfsuite')

#         # tag sentences in the dev set
#         y_pred_dev = [tagger.tag(xseq) for xseq in trial_dev_x]
#         perf = performance(y_dev, y_pred_dev)
#         macro_F1 = perf['macro_F1']
#         if macro_F1 > BEST_PERFORMANCE_SO_FAR:
#             BEST_PERFORMANCE_SO_FAR = macro_F1
#             BEST_SELECTION_SO_FAR = selected_feats

#         for feat in selected_feats:
#             FEATURE_SCORES[feat] += macro_F1

#     rank = sorted([(k,v) for k,v in FEATURE_SCORES.items()], key=lambda x: x[1], reverse=True)
#     print('feats =', [i[0] for i in rank[:20]])
#     TRIAL_APPEARANCES.append(rank[:20])

# print('best score', BEST_PERFORMANCE_SO_FAR, BEST_SELECTION_SO_FAR) 
# appearance_count = sum(TRIAL_APPEARANCES, [])

# a = {}
# for i in [j[0] for j in appearance_count]:
#     if i not in a: a[i] = 1
#     else: a[i] += 1
# a = sorted([(k,v) for k,v in a.items()], key=lambda x: x[1], reverse=True)
# [i[0] for i in a[:10]]  # print top 10 features so can make report on them without over 2 pages 

/tmp/ipykernel_3423/3611008565.py:87: RuntimeWarning: invalid value encountered in scalar divide
  macro_P = np.array([confusion_matrix[i,i]/confusion_matrix[:,i].sum() for i in range(len(LABELS))])


feats = ['rat', '-1:word', 'stemgloss', 'BOS', '+1:word', 'gloss', 'prc3', 'd3tok', 'd1seg', 'root', 'd2tok', 'bwtok', 'stem', 'atbseg', 'catib6', 'diac', 'EOS', 'd1tok', 'gen', 'cas']
feats = ['-1:word', 'stem', 'stemgloss', 'root', 'ud', 'rat', 'lex', 'mod', 'd2seg', 'pattern', 'catib6', 'd1tok', 'source', 'd3seg', 'bw', 'prc1', 'd1seg', 'form_gen', 'gloss', 'atbseg']
feats = ['rat', 'pos_logprob', '-1:word', 'd1tok', 'cas', 'prc2', 'atbtok', 'stem', 'bw', 'd2seg', 'stemcat', 'vox', 'diac', 'BOS', 'lex', '+1:word', 'form_gen', 'word', 'd1seg', 'stemgloss']


/tmp/ipykernel_3423/3611008565.py:89: RuntimeWarning: invalid value encountered in divide
  macro_F1 = 2*macro_P*macro_R/(macro_P + macro_R)


feats = ['catib6', 'prc2', 'rat', 'num', 'per', 'd2tok', 'pos_logprob', 'ud', 'gen', 'enc0', 'BOS', 'prc3', 'root', 'pos', 'caphi', 'EOS', 'bw', 'stem', 'gloss', 'form_gen']
feats = ['rat', 'stem', 'gloss', 'source', '-1:word', 'd2tok', 'form_gen', 'pos_lex_logprob', 'vox', 'EOS', 'word', 'pos', 'num', 'stemgloss', '+1:word', 'pos_logprob', 'lex_logprob', 'lex', 'per', 'd3seg']
feats = ['-1:word', 'pos_logprob', 'pos', 'form_num', 'stt', 'atbseg', '+1:word', 'stemcat', 'stemgloss', 'EOS', 'root', 'd2seg', 'd3tok', 'gen', 'd3seg', 'enc0', 'source', 'prc1', 'pattern', 'stem']
feats = ['pattern', 'bwtok', 'stem', '-1:word', 'lex', 'stemgloss', 'prc1', 'prc2', 'stemcat', 'ud', 'diac', 'bw', 'pos_lex_logprob', 'd1seg', 'pos', 'root', 'atbtok', 'cas', 'EOS', 'catib6']
feats = ['rat', 'stt', '+1:word', 'lex', 'atbtok', '-1:word', 'd1seg', 'd3seg', 'stem', 'stemcat', 'cas', 'prc2', 'prc0', 'gen', 'mod', 'catib6', 'gloss', 'vox', 'lex_logprob', 'd1tok']
feats = ['rat', 'atbtok', 'asp', 'mod', '

['atbseg',
 'diac',
 'mod',
 'source',
 'prc1',
 'vox',
 'num',
 'per',
 'enc0',
 'pos_lex_logprob',
 'd3tok',
 'caphi',
 'lex_logprob',
 'form_num',
 'stt',
 'prc0',
 'asp']

In [37]:
{2,3,4}-{None}

{2, 3, 4}

Measuring impact of each features

In [ ]:
# This does not help, still around 80 on dev 
import pycrfsuite

FEATURE_SCORES = {k: 0. for k in ALL_FEATURES}
for feat in list(FEATURE_SCORES.keys()) + [None]:
    KEYS = list(ALL_FEATURES - {feat}) if feat is not None else ALL_FEATURES
    X_train = [sent2features(s) for s in tqdm(train_sents[:500])]
    y_train = [sent2labels(s) for s in train_sents[:500]]
    X_dev = [sent2features(s) for s in tqdm(dev_sents)]
    y_dev = [sent2labels(s) for s in dev_sents]

    # training: copy from below
    hyper_trainer = pycrfsuite.Trainer(verbose=False)
    for xseq, yseq in zip(X_train, y_train):
        hyper_trainer.append(xseq, yseq)
    hyper_trainer.set_params({
        'c1': 1.0,   # coefficient for L1 penalty
        'c2': 1e-3,  # coefficient for L2 penalty
        'max_iterations': 50,  # stop earlier
        'feature.possible_transitions': True
    })
    hyper_trainer.train('arabic-ner.crfsuite')

    # dev eval
    tagger = pycrfsuite.Tagger()
    tagger.open('arabic-ner.crfsuite')

    # tag sentences in the dev set
    y_pred_dev = [tagger.tag(xseq) for xseq in X_dev]
    perf = performance(y_dev, y_pred_dev)
    macro_F1 = perf['macro_F1']
    FEATURE_SCORES[feat] = macro_F1

rank = sorted([(k,v) for k,v in FEATURE_SCORES.items()], key=lambda x: x[1], reverse=True)
[i[0] for i in rank[:10]] # print top 10 features so can make report on them without over 2 pages 

100%|██████████| 398/398 [00:01<00:00, 268.68it/s]
/var/folders/kx/8y58jc1x01z8bx9qp5s5ktl00000gp/T/ipykernel_87129/2039550972.py:82: RuntimeWarning: invalid value encountered in scalar divide
  macro_P = np.array([confusion_matrix[i,i]/confusion_matrix[:,i].sum() for i in range(len(LABELS))])


['root', 'asp', 'pos', 'vox', 'stt', 'num', 'd1seg', 'd2tok', 'd1tok', 'd3tok']

In [48]:
[i[0] for i in rank[-20:-1]]

['pos_lex_logprob',
 'gen',
 'stemgloss',
 'form_gen',
 'lex',
 'stem',
 'source',
 'pattern',
 'prc0',
 'stemcat',
 'prc3',
 'prc2',
 'caphi',
 '+1:word',
 'cas',
 'ud',
 'per',
 '-1:word',
 'rat']

Extract the features from the data and train your model.

In [49]:
import pycrfsuite

KEYS = ['pos_lex_logprob',
 'gen',
 'stemgloss',
 'form_gen',
 'lex',
 'stem',
 'source',
 'pattern',
 'prc0',
 'stemcat',
 'prc3',
 'prc2',
 'caphi',
 '+1:word',
 'cas',
 'ud',
 'per',
 '-1:word',
 'rat']

# create pycrfsuite.Trainer instance
trainer = pycrfsuite.Trainer(verbose=False)

X_train = [sent2features(s) for s in tqdm(train_sents)]
y_train = [sent2labels(s) for s in train_sents]
LABELS = sorted(list(set(sum(y_train, []))))

X_dev = [sent2features(s) for s in tqdm(dev_sents)]
y_dev = [sent2labels(s) for s in dev_sents]

# load the training data
for xseq, yseq in zip(X_train, y_train):
    trainer.append(xseq, yseq)

# training parameteres
trainer.set_params({
    'c1': 1.0,   # coefficient for L1 penalty
    'c2': 1e-3,  # coefficient for L2 penalty
    'max_iterations': 50,  # stop earlier
    # include transitions that are possible, but not observed
    'feature.possible_transitions': True
})

trainer.train('arabic-ner.crfsuite')


100%|██████████| 398/398 [00:01<00:00, 273.21it/s]


## **Evaluation**

Evaluate your model on the development set and the test set.

**Note**: You may run evaluation on the test set **ONLY ONCE** at the end. You should not use the test set to check how well your model is currently performing and modify the model accordingly.
Use the dev set, or create a tune set for this purpose.

In [50]:
# instantiate the trained tagger
tagger = pycrfsuite.Tagger()
tagger.open('arabic-ner.crfsuite')

# tag sentences in the dev set
y_pred_dev = [tagger.tag(xseq) for xseq in X_dev]

In [51]:
performance(y_dev, y_pred_dev)

{'micro_avg': np.float64(0.9739483747609943),
 'macro_P': np.float64(0.887663727863919),
 'macro_R': np.float64(0.750510765129192),
 'macro_F1': np.float64(0.8037294734148466)}

In [52]:
X_test = [sent2features(s) for s in tqdm(test_sents)]
y_test = [sent2labels(s) for s in test_sents]
y_pred_test = [tagger.tag(xseq) for xseq in X_test]
performance(y_test, y_pred_test)

100%|██████████| 925/925 [00:02<00:00, 321.92it/s]


{'micro_avg': np.float64(0.9389795265515035),
 'macro_P': np.float64(0.7393141829709573),
 'macro_R': np.float64(0.5823860962774592),
 'macro_F1': np.float64(0.6364313471865182)}

# Export

In [60]:
file_name = 'ANERCorp_CamelLab_test.pred.txt'
with open(file_name, 'w') as wf:
    for sent_x, sent_y in zip(test_sents, y_pred_test):
        for word, label in zip(sent_x, sent_y):
            wf.write(f'{word[0]} {label}\n')
        wf.write('\n')